# Notebook 07: Evaluación Cuantitativa y Métricas de Rendimiento

## Contexto y Rigor Analítico
Auditoría y evaluación cuantitativa formal del rendimiento de los modelos desarrollados sobre el conjunto de test independiente:
- Matrices de confusión multiclase.
- Curvas de Precisión vs. Recall y F1-Score por categoría.
- Análisis de casos borde (*edge cases*) y tasa de falsos positivos en vehículos sin daños.


In [26]:
import json 
import keras
import torch
import config
import cv2 as cv
import pandas as pd

from pipeline.preprocessing.cocoDataloader import COCODataset

import torchvision
from torch.optim import SGD
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import load_model

# Cargamos los modelos

In [16]:
KerasModelPath = config.CLASSIFICATION_MODEL_PATH
KerasLabelsPath = config.CLASSIFICATION_LABEL_PATH
DetectorModelPath = config.OBJECT_DETECTION_MODEL_PATH
DetectorLabelsPath = config.OBJECT_DETECTION_LABEL_PATH

In [17]:
KerasModel = load_model(KerasModelPath)
KerasLabels = pd.read_csv(KerasLabelsPath)['description'].values

In [18]:
with open(DetectorLabelsPath, 'r') as archivo:
    json_data = json.load(archivo)
    class_mappings = {item['id']: item['name'] for item in json_data['classes']}
    TorchLabels = class_mappings

In [19]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
num_classes = len(TorchLabels) + 1
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
TorchOptimizer = SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
TorchModels = model

In [20]:
TorchModels.load_state_dict(torch.load(DetectorModelPath, weights_only=True))
TorchModels.eval()

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [21]:
TorchTransform = transforms.Compose([
    transforms.ToTensor()
])
TorchDevice = torch.device("cpu")

# Cargamos los datos con los que vamos a calcular las métricas

In [22]:
Val_Data_Keras = F"{config.PROCESSED_PATH_ORIGINAL}/csv/val.csv"
Val_Data_PyTorch = config.PROCESSED_PATH_ORIGINAL

In [27]:
def cargar_datos(Torch_Path_Validation, Keras_Path_Validation):
    torch_val = COCODataset(
        json_file=F"{Torch_Path_Validation}/json/test.json", 
        root_dir=F"{Torch_Path_Validation}/car-parts/test", 
        transforms=TorchTransform)
    
    torch_loader = DataLoader(torch_val, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

    df = pd.read_csv(Keras_Path_Validation)
    
    x_val_csv =[]
    for img in df['image_path']:
        imagen = cv.imread(img)
        x_val_csv.append(cv.resize(imagen,(180,180)))
    y_val_csv = df['damaged']        
        
    y_val_encoded = to_categorical(y_val_csv)
    
    X_keras = tf.convert_to_tensor(x_val_csv)
    y_keras = tf.convert_to_tensor(y_val_encoded)
    
    return torch_loader, X_keras, y_keras

In [28]:
Val_Torch, X_keras, y_keras = cargar_datos(Val_Data_PyTorch, Val_Data_Keras)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


# Tenemos cargados los datos

In [45]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt

In [46]:
KerasCheckpointPath = config.CLASSIFICATION_CHECKPOINT_PATH
KerasCheckpointList = []
for epoch in range(1, 26): 
    path = f"./models/classification/checkpoints/save_at_{epoch}.keras"
    KerasCheckpointList.append(path)

In [51]:
TorchCheckpointPath = config.OBJECT_DETECTION_CHECKPOINT_PATH
TorchCehckpointList = []
for epoch in range(1, 26): 
    path = f"./models/object_detection/checkpoints/checkpoint-{epoch}.keras"
    TorchCehckpointList.append(path)

In [47]:
def evaluate_keras_model(checkpoint_path, X_keras, y_keras):
    kerasCheckpointModel = load_model(checkpoint_path)
    preds = kerasCheckpointModel.predict(X_keras)

    # Calcula ROC-AUC y PR-ROC
    roc_auc_scores = {}
    pr_aucs = {}
    for i in range(y_keras.shape[1]):
        fpr, tpr, _ = roc_curve(y_keras[:, i], preds[:, i])
        precision, recall, _ = precision_recall_curve(y_keras[:, i], preds[:, i])
        roc_auc_scores[i] = auc(fpr, tpr)
        pr_aucs[i] = auc(recall, precision)
        
        plt.figure()
        plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc_scores[i]:.2f})')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend()
        plt.title("ROC Curve")
        plt.show()

    # Matriz de confusión
    y_pred_labels = np.argmax(preds, axis=1)
    y_true_labels = np.argmax(y_keras, axis=1)
    cm = confusion_matrix(y_true_labels, y_pred_labels)
    print("Confusion Matrix:\n", cm)

In [50]:
for checkpoint in KerasCheckpointList:
    # evaluate_keras_model(checkpoint, X_keras, y_keras)
    print(checkpoint)

./models/classification/checkpoints/save_at_1.keras
./models/classification/checkpoints/save_at_2.keras
./models/classification/checkpoints/save_at_3.keras
./models/classification/checkpoints/save_at_4.keras
./models/classification/checkpoints/save_at_5.keras
./models/classification/checkpoints/save_at_6.keras
./models/classification/checkpoints/save_at_7.keras
./models/classification/checkpoints/save_at_8.keras
./models/classification/checkpoints/save_at_9.keras
./models/classification/checkpoints/save_at_10.keras
./models/classification/checkpoints/save_at_11.keras
./models/classification/checkpoints/save_at_12.keras
./models/classification/checkpoints/save_at_13.keras
./models/classification/checkpoints/save_at_14.keras
./models/classification/checkpoints/save_at_15.keras
./models/classification/checkpoints/save_at_16.keras
./models/classification/checkpoints/save_at_17.keras
./models/classification/checkpoints/save_at_18.keras
./models/classification/checkpoints/save_at_19.keras
./

In [ ]:
def evaluate_pytorch_model(checkpoint_path, dataloader, coco):
    model = TorchModels.load_state_dict(torch.load(DetectorModelPath, weights_only=True))
    TorchModels.eval()
    model.eval()
    all_detections = []

    with torch.no_grad():
        for images, targets in dataloader:
            predictions = model(images)
            for i, prediction in enumerate(predictions):
                image_id = targets[i]["image_id"].item()
                for bbox, score, label in zip(prediction["boxes"], prediction["scores"], prediction["labels"]):
                    bbox = bbox.cpu().numpy().tolist()
                    score = score.item()
                    label = label.item()
                    all_detections.append({
                        "image_id": image_id,
                        "category_id": label,
                        "bbox": bbox,
                        "score": score
                    })

    coco_detections = coco.loadRes(all_detections)
    coco_eval = COCOeval(coco, coco_detections, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()


In [2]:
print(1 - (0.00178362250328064 + 0.0053612530231476))


0.9928551244735717
